<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>Dataset과 DataLoader(Data Pipeline)</strong>에 대해 학습합니다.</br></br>
>PyTorch의 데이터 파이프라인을 구성하고 배치 단위 학습을 학습해봅시다.

</br>

# Dataset
> PyTorch에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">데이터를 추상화</mark>하는 기본 클래스입니다.

> 학습 데이터를 한꺼번에 메모리에 올리면 ImageNet처럼 수백 GB에 달하는 데이터셋에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">메모리 부족(OOM)</mark> 오류가 발생합니다.</br></br>
> Dataset 추상화는 `__getitem__`으로 필요한 샘플만 그때그때 불러오는 지연 로딩(lazy loading)을 가능하게 합니다. 또한 샘플을 하나씩 모델에 넣으면 GPU 병렬 연산을 활용하지 못하는데, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">DataLoader</mark>는 여러 샘플을 배치 텐서로 묶어 GPU 처리 효율을 극대화합니다.</br></br>
> `shuffle=True`로 매 에폭마다 순서를 무작위로 섞어 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">일반화 성능을 향상</mark>시키고, `num_workers > 0` 설정으로 데이터 로딩을 별도 프로세스에서 미리 수행하여 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">파이프라인 병렬성</mark>을 실현할 수 있습니다.

## TensorDataset
> 텐서를 직접 감싸는 가장 간단한 Dataset입니다.

In [ ]:
# TODO 1: X에 [[1,2],[3,4],[5,6],[7,8]]을 실수형 텐서로, y에 [0,1,0,1]을 정수형 텐서로 생성한 뒤, 텐서 데이터셋으로 묶어 길이와 첫 번째 샘플을 출력해봅시다.

</br>

## 커스텀 Dataset
> `__len__`과 `__getitem__`을 구현하여 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">자유로운 데이터 로딩</mark>이 가능합니다.

In [ ]:
# TODO 2: 커스텀 데이터셋 클래스를 정의하고, 초기화에서 X를 실수형, y를 정수형 텐서로 변환하여 저장하고, 데이터 개수 반환과 인덱스로 샘플을 가져오는 메서드를 구현해봅시다.


class MyDataset(Dataset):
    def __init__(self, X, y):

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

</br>

# DataLoader
> Dataset을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배치 단위로 순회</mark>할 수 있게 하는 반복자(Iterator)입니다.

In [ ]:
# TODO 3: 데이터 로더를 import하고, dataset을 batch_size=2, shuffle=True로 감싼 뒤, 반복문으로 각 배치의 shape과 라벨을 출력해봅시다.

</br>

## DataLoader 주요 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th style="text-align:center">설명</th>
      <th style="text-align:center">기본값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>batch_size</code></td><td style="text-align:center">배치 크기</td><td style="text-align:center"><code>1</code></td></tr>
    <tr><td style="text-align:center"><code>shuffle</code></td><td style="text-align:center">에폭마다 순서 섞기</td><td style="text-align:center"><code>False</code></td></tr>
    <tr><td style="text-align:center"><code>num_workers</code></td><td style="text-align:center">병렬 데이터 로딩 프로세스 수</td><td style="text-align:center"><code>0</code></td></tr>
    <tr><td style="text-align:center"><code>drop_last</code></td><td style="text-align:center">마지막 불완전 배치 버리기</td><td style="text-align:center"><code>False</code></td></tr>
    <tr><td style="text-align:center"><code>pin_memory</code></td><td style="text-align:center">GPU 전송 최적화</td><td style="text-align:center"><code>False</code></td></tr>
  </tbody>
</table>

💡shuffle 설정
> 학습용: `shuffle=True` (매 에폭 순서 변경으로 일반화 향상)</br>
> 검증/테스트용: `shuffle=False` (재현성 보장)

</br>

## 학습 루프에서의 활용

In [ ]:
# TODO 4: 간단한 모델을 만들고, 학습 루프에서 DataLoader를 활용해봅시다.


# 간단한 선형 모델 (입력 2 → 출력 2)

💡batch_size와 메모리
> 배치 크기가 클수록 GPU 메모리를 많이 사용합니다.</br>
> `RuntimeError: CUDA out of memory`가 발생하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배치 크기를 줄이세요</mark>.